# Ensembles beyond trees

[Random forests](../random_forests.ipynb) average trees, and [gradient boosting](../gradient_boosting.ipynb) adds them in sequence. Neither idea is limited to trees: any models can be averaged, combined by a model trained for the purpose, or bagged. Here the three are tried on California housing, measured on the validation rows of [Validation and model selection](../../../ml-fundamentals/validation_and_model_selection.ipynb), and finds when combining helps, and when it doesn't.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns="MedHouseVal")
y = 100_000 * housing["MedHouseVal"]
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_validation, y_train, y_validation = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)


def validation_mae(model):
    return round(mean_absolute_error(y_validation, model.fit(X_train, y_train).predict(X_validation)))

## Three different models

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

members = {
    "linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "k-nearest neighbours, distance weights": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10, weights="distance")),
    "gradient boosting": HistGradientBoostingRegressor(random_state=42),
}
{name: validation_mae(model) for name, model in members.items()}

{'linear regression': 53333,
 'k-nearest neighbours, distance weights': 43924,
 'gradient boosting': 32175}

## Voting: the average of the predictions

`VotingRegressor` trains each model and predicts the average of their predictions (`VotingClassifier` takes the majority vote, or averages the probabilities):

In [ ]:
from sklearn.ensemble import RandomForestRegressor, VotingRegressor

{
    "the three models": validation_mae(VotingRegressor(list(members.items()))),
    "boosting and a random forest": validation_mae(VotingRegressor([
        ("gradient boosting", HistGradientBoostingRegressor(random_state=42)),
        ("random forest", RandomForestRegressor(random_state=42, n_jobs=-1)),
    ])),
}

{'the three models': 39421, 'boosting and a random forest': 31960}

Averaging the three gives about \$39,400, worse than gradient boosting alone (\$32,200): every prediction is pulled towards the larger errors of the linear model and the neighbours. Averaging boosting with a random forest, two strong models built in different ways, gives about \$32,000, a little better than either (the forest alone scored about \$33,600 in [Validation and model selection](../../../ml-fundamentals/validation_and_model_selection.ipynb)). The gain is small, and here it came only from members that are good to begin with.

## Stacking: a model that learns how to combine

Instead of a plain average, **stacking** trains a second model, the *final estimator*, to combine the members' predictions. To train it on predictions the members didn't make on their own training rows, `StackingRegressor` uses cross-validated predictions (`cv=5`); a Ridge regression then learns one weight per member:

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV

stacking = StackingRegressor(list(members.items()), final_estimator=RidgeCV(), cv=5)
validation_mae(stacking), dict(zip(members, stacking.final_estimator_.coef_.round(2)))

(32133,
 {'linear regression': np.float64(-0.0),
  'k-nearest neighbours, distance weights': np.float64(0.01),
  'gradient boosting': np.float64(1.0)})

The final estimator gave gradient boosting a weight of 1.0 and the other two almost nothing: it learned, from cross-validated predictions, that the best use of the weak members is to ignore them. The result, about \$32,100, is practically boosting's own (\$32,200). Stacking only reweighs what the members predict: with one strong member and two weak ones, the best weighting keeps the strong one, and avoids the loss that plain voting suffered. With strong members that make different errors, it can beat each of them.

## Bagging any model

[Bagging](../random_forests.ipynb#many-trees-on-resampled-data) trains each copy of a model on a bootstrap sample and averages them; `BaggingRegressor` does it for any model. On a single decision tree, and on k-nearest neighbours:

In [ ]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor

neighbours = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10))
{
    "one tree": validation_mae(DecisionTreeRegressor(random_state=42)),
    "100 bagged trees": validation_mae(BaggingRegressor(DecisionTreeRegressor(random_state=42), n_estimators=100,
                                                        random_state=42, n_jobs=-1)),
    "k-nearest neighbours, uniform weights": validation_mae(neighbours),
    "30 bagged k-nearest neighbours": validation_mae(BaggingRegressor(neighbours, n_estimators=30, random_state=42, n_jobs=-1)),
}

{'one tree': 45594,
 '100 bagged trees': 33497,
 'k-nearest neighbours, uniform weights': 44462,
 '30 bagged k-nearest neighbours': 43622}

Bagging turns one tree's \$45,600 into \$33,500, a quarter less: a single deep tree changes a lot with its training rows, and averaging bootstrapped copies cancels much of that. The same bagging barely moves k-nearest neighbours, from \$44,500 to \$43,600: its predictions are already averages over ten neighbours, so resampling the rows changes them little. Bagging is a cure for variance, and only helps where there is variance to cure. With scikit-learn's defaults for regression, a hundred bagged trees *are* a random forest: the random choice of features at each split only starts when `max_features` is below 1.

## When combining helps

An ensemble improves on its members when they are **good** and make **different** errors. Averaging a strong model with weak ones drags it down; averaging models that make the same errors changes nothing; averaging strong models that fail in different places is where the gain is. Bagging creates that difference artificially, by resampling, and so helps models that change a lot with their data, like deep trees, and hardly models that don't, like k-NN or linear regression. Stacking can learn which members to trust, at the cost of more training and one more model to validate.

In competitions, ensembles of several boosted models and networks often win by small margins. In practice, each member is one more model to train, monitor and explain, and a gain of a few hundred dollars of error has to be worth that.